
# MODULE 35.9 — AUDIO AUGMENTATION FOR SER

**Must Know**

Audio augmentation is an important step before model training.

We have learned how to extract features from audio, but our dataset may contain limited recordings, different speakers, and different recording conditions.

Audio augmentation helps us create realistic variations of existing training recordings so that our model doesn't simply memorize the original samples.

But there's one major difference between ordinary image augmentation and audio augmentation:

**An audio transformation can accidentally change the emotion being expressed.**

For example, increasing pitch too much might make a sad recording sound excited or unnatural.

So we'll focus on *emotion-preserving augmentation*.

---

## 1. Why do we need audio augmentation?

Imagine our SER dataset contains:

| Emotion | Number of recordings |
| ------- | -------------------- |
| Happy   | 500                  |
| Sad     | 450                  |
| Angry   | 120                  |
| Neutral | 480                  |

There are two potential problems:

1. The dataset may contain limited acoustic variation.
2. The number of recordings may be imbalanced across emotion classes.

A machine-learning model could overfit to the specific recordings instead of learning generalizable emotional characteristics.

### What is audio augmentation?

Audio augmentation is the process of applying controlled transformations to an existing recording to generate additional training examples.

For example:

**Original audio**

Happy speech recording

**Augmentation operations**

- Volume variation.
- Background noise.
- Time stretching.
- Pitch shifting.

**Augmented recordings**

Different acoustic versions with the original emotion label retained only when justified.

### Why is it useful for SER?

Augmentation can help a model become less sensitive to:

- Small changes in recording volume.
- Background noise.
- Slight differences in speaking speed.
- Small variations in pitch.
- Differences in recording alignment.

**Important:** Augmentation does not create genuinely new emotional experiences or new independent speakers. It creates transformed versions of existing recordings.

---

## 2. Background-noise addition

**Must Know**

Real-world speech recordings often contain background noise.

Examples:

- Traffic.
- Fan noise.
- Air conditioning.
- Keyboard sounds.
- Room ambience.
- Recording-device noise.

We can simulate some of these conditions by adding noise to the original waveform.

### Mathematical representation

Let:

- $x[n]$ = original speech.
- $v[n]$ = background noise.
- $y[n]$ = augmented speech.

Then:

$$
y[n]=x[n]+\alpha v[n]
$$

Here, $\alpha$ controls the noise amplitude.

However, choosing an arbitrary noise amplitude is not ideal. We should control the signal-to-noise ratio (SNR).

### What is SNR?

Signal-to-noise ratio describes the relative power of the desired signal and the added noise.

$$
SNR_{dB}
=
10\log_{10}
\left(
\frac{P_{signal}}{P_{noise}}
\right)
$$

For example:

| SNR | Interpretation |
|---|---|
| 30 dB | Relatively clean |
| 20 dB | Mild noise |
| 10 dB | More noticeable noise |
| 0 dB | Signal and noise have equal power |

These are illustrative conditions; perceived noise also depends on the type of noise.

### Why is this useful for SER?

Suppose we train our model only on clean recordings.

When it encounters speech recorded near a fan or in a busy environment, its predictions may become less reliable.

Noise augmentation exposes the model to controlled recording variations.

**Caution:** Excessive noise can hide emotional characteristics, making the augmented recording unsuitable for retaining its original emotion label.

---

## 3. Small volume variations

**Must Know**

Volume augmentation changes the amplitude of an audio recording without intentionally changing its timing or pitch.

Mathematically:

$$
y[n]=g\cdot x[n]
$$

Where:

- $x[n]$ = original waveform.
- $g$ = amplitude gain.
- $y[n]$ = output waveform.

For example:

- $g=0.8$: lower amplitude.
- $g=1.0$: unchanged.
- $g=1.2$: higher amplitude.

### Why use it?

The same speaker may sound louder or quieter because of:

- Microphone distance.
- Microphone sensitivity.
- Recording gain.
- Environmental conditions.

Small gain variations help the model avoid relying entirely on absolute recording amplitude.

### Important limitation

Volume can itself carry emotional information.

For example, forceful speech and soft speech may differ in vocal intensity.

If we apply excessive volume normalization or augmentation, we might remove useful emotional cues.

For our SER project, use small gain changes rather than aggressive normalization.

---

## 4. Time shifting

**Must Know**

Time shifting moves the audio waveform forward or backward in time.

For example:

**Original:**

```text
| Speech begins | Speech | Silence |
```

**Shifted:**

```text
| Silence | Speech begins | Speech |
```

Mathematically, a simple time shift is:

$$
y[n]=x[n-k]
$$

Where $k$ represents the number of samples shifted.

### Why is it useful?

It helps the model become less sensitive to the exact position of speech within an audio segment.

For example, two recordings may contain the same utterance but begin at slightly different positions.

### Important implementation detail

A circular shift using `np.roll()` moves samples from the end of the recording to the beginning.

This can create an unrealistic discontinuity.

For speech augmentation, a safer approach is to shift the waveform and fill the exposed region with zeros.

---

## 5. Time stretching

**Must Know**

Time stretching changes the duration of a recording without intentionally changing its pitch.

For example:

Original speech duration:

$$
T=4\text{ seconds}
$$

After stretching:

- Slower version: 5 seconds.
- Faster version: 3.2 seconds.

The pitch should remain approximately the same.

### Why is it useful?

People speak at different rates.

Even the same speaker can change their speaking speed.

Time stretching introduces controlled variations in speaking rate.

### Important distinction

Time stretching is not the same as changing the playback sample rate.

If we simply play a recording at a different sample rate, both its duration and pitch change.

A proper time-stretching algorithm attempts to change duration while preserving pitch.

### SER caution

Speaking rate can be an important emotional feature.

Aggressive stretching may change the perceived emotional delivery.

We'll use small variations.

---

## 6. Pitch shifting

**Must Know**

Pitch shifting changes the perceived pitch of an audio recording while attempting to preserve its duration.

For example:

- Original: 150 Hz.
- Shifted upward: approximately 165 Hz.
- Shifted downward: approximately 135 Hz.

The exact transformation depends on the algorithm and the signal.

Pitch shifting is often expressed in semitones.

One semitone corresponds to a frequency ratio of:

$$
2^{1/12}
$$

For a shift of $s$ semitones:

$$
f_{new}=f_{original}\times2^{s/12}
$$

### Why is it useful?

It introduces small variations in vocal pitch and can reduce sensitivity to specific pitch patterns.

### Why is it risky for SER?

Pitch is itself an important emotion-related feature.

A sad utterance shifted too far upward may sound less sad or unnatural.

Therefore:

- Use small pitch shifts.
- Listen to the transformed recording.
- Compare F0 before and after.
- Retain the original emotion label only if the emotion remains recognizable.

---

## 7. Random cropping

**Must Know**

Random cropping selects a shorter segment from a longer recording.

Suppose our original recording is 5 seconds long.

We want a 3-second segment.

Instead of always selecting the first 3 seconds, we randomly choose a starting position.

Mathematically:

$$
y[n]=x[n+k]
$$

Where $k$ is the randomly selected starting sample.

### Why is it useful?

It helps the model learn from different portions of an utterance rather than always seeing the same beginning and ending.

### Important SER caution

Emotion can change throughout a recording.

For example:

- The speaker starts neutrally.
- Becomes angry halfway through.
- Ends with a shouting expression.

A random crop might contain only the neutral portion.

But if we retain the original angry label, we introduce a potentially incorrect training example.

Therefore:

- Avoid extremely short crops.
- Preserve enough emotional context.
- Use utterance-level labels only when the cropped section still represents that emotion.
- Consider segment-level labels when emotion changes within recordings.

---

## 8. Augmentation strength and realism

**Must Know**

This is probably the most important practical consideration.

We can apply all the transformations correctly from a programming perspective and still generate bad training data.

Consider an angry recording.

| Transformation | Mild version | Excessive version |
|---|---|---|
| Noise addition | Slight room noise | Speech becomes difficult to understand |
| Volume | Small gain variation | Distorted or clipped waveform |
| Time stretching | Slightly slower | Unnatural speaking rhythm |
| Pitch shifting | Small pitch change | Different perceived voice character |
| Time shifting | Small alignment change | Important speech removed |
| Random cropping | Retains emotional expression | Removes the emotion-bearing segment |

### What is emotion preservation?

An augmented recording should ideally preserve the emotion represented by its original label.

For example:

$$
Original: Angry
$$

$$
Augmented: Angry
$$

If an augmentation changes the perceived emotion:

$$
Original: Angry
$$

$$
Augmented: Neutral
$$

Then retaining the original label would introduce label noise.

**Our rule:** Every augmentation must be acoustically plausible and sufficiently emotion-preserving.

---

## 9. Why augmentation must only be applied to training data

**Critical ML concept**

This is related to data leakage.

Suppose we divide our dataset into:

- Training: 70%.
- Validation: 15%.
- Testing: 15%.

The correct workflow is:

**Original dataset**

↓

**Split into train / validation / test**

↓

**Training**

Augment

**Validation**

Original

**Testing**

Original

### Why?

Imagine that we augment all recordings before splitting.

The original recording might enter the training set, while its augmented version enters the testing set.

The model has already seen almost the same recording during training.

This can produce misleadingly high test performance.

### Correct approach

1. Split the original recordings first.
2. Apply augmentation only to training recordings.
3. Keep validation and testing recordings unchanged.
4. Evaluate the model on unseen original recordings.

**Additional SER consideration:** If multiple recordings belong to the same speaker, use speaker-independent splitting where possible. Otherwise, the model might recognize speaker characteristics rather than generalizable emotional patterns.



# Practical Lab 35H — Safe Augmentation

**Hands-on implementation**

Our goal is to create original and augmented versions of selected training recordings and compare their acoustic characteristics.

We'll implement:

1. Background-noise addition.
2. Volume variation.
3. Time shifting.
4. Time stretching.
5. Pitch shifting.
6. Random cropping.

We'll also listen to the recordings and inspect their waveforms, pitch and energy.

---

## Step 1 — Import libraries

Install the required libraries:

```python
%pip install librosa soundfile numpy pandas matplotlib
```

Import them:

```python
import librosa
import librosa.display

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import soundfile as sf

from pathlib import Path
from IPython.display import Audio, display
```

---

## Step 2 — Configure parameters

```python
SR = 16000

N_FFT = 512
WIN_LENGTH = 400
HOP_LENGTH = 160

AUGMENTATION_SEED = 42

rng = np.random.default_rng(
    AUGMENTATION_SEED
)
```

Load a training recording:

```python
audio, sr = librosa.load(
    "audio/train/sample.wav",
    sr=SR,
    mono=True
)

print("Sample rate:", sr)
print("Duration:", len(audio) / sr)
```

---

## Step 3 — Create a background-noise augmentation function

We'll control noise using SNR rather than choosing an arbitrary amplitude.

```python
def add_background_noise(
    audio,
    snr_db=25,
    rng=None
):

    if rng is None:
        rng = np.random.default_rng()

    noise = rng.normal(
        0,
        1,
        size=len(audio)
    )

    signal_power = np.mean(audio ** 2)
    noise_power = np.mean(noise ** 2)

    if signal_power < 1e-12:
        return audio.copy()

    target_noise_power = (
        signal_power / (10 ** (snr_db / 10))
    )

    noise = noise * np.sqrt(
        target_noise_power / noise_power
    )

    augmented = audio + noise

    return augmented
```

### Usage

```python
noisy_audio = add_background_noise(
    audio,
    snr_db=25,
    rng=rng
)
```

### Try a few SNR values

```python
noise_30 = add_background_noise(
    audio,
    snr_db=30,
    rng=rng
)

noise_20 = add_background_noise(
    audio,
    snr_db=20,
    rng=rng
)

noise_10 = add_background_noise(
    audio,
    snr_db=10,
    rng=rng
)
```

### Listen to them

```python
print("Original")
display(Audio(audio, rate=sr))

print("30 dB SNR")
display(Audio(noise_30, rate=sr))

print("20 dB SNR")
display(Audio(noise_20, rate=sr))

print("10 dB SNR")
display(Audio(noise_10, rate=sr))
```

Listen carefully.

At what point does the noise start interfering with the emotional characteristics?

**Note:** This function generates synthetic white noise. It is useful for learning and an initial experiment, but it is not equivalent to adding real traffic, fan or room recordings. Later, we can use real background-noise recordings for more realistic augmentation.

---

## Step 4 — Create a volume variation function

```python
def change_volume(
    audio,
    min_gain=0.85,
    max_gain=1.15,
    rng=None
):

    if rng is None:
        rng = np.random.default_rng()

    gain = rng.uniform(
        min_gain,
        max_gain
    )

    augmented = audio * gain

    return augmented, gain
```

### Usage

```python
volume_audio, gain = change_volume(
    audio,
    rng=rng
)

print("Applied gain:", gain)

display(Audio(audio, rate=sr))
display(Audio(volume_audio, rate=sr))
```

### Check for clipping

```python
print(
    "Original peak:",
    np.max(np.abs(audio))
)

print(
    "Augmented peak:",
    np.max(np.abs(volume_audio))
)
```

The waveform should not exceed the intended amplitude range.

Avoid silently clipping it, because clipping changes the waveform and introduces distortion.

---

## Step 5 — Create a time-shifting function

We'll shift the waveform and fill the exposed region with zeros.

```python
def time_shift(
    audio,
    shift_seconds,
    sr
):

    shift_samples = int(
        shift_seconds * sr
    )

    augmented = np.zeros_like(audio)

    # Positive shift: move audio to the right
    if shift_samples > 0:

        if shift_samples < len(audio):

            augmented[shift_samples:] = (
                audio[:-shift_samples]
            )

    # Negative shift: move audio to the left
    elif shift_samples < 0:

        shift_samples = abs(shift_samples)

        if shift_samples < len(audio):

            augmented[:-shift_samples] = (
                audio[shift_samples:]
            )

    else:

        augmented = audio.copy()

    return augmented
```

### Usage

```python
shifted_audio = time_shift(
    audio,
    shift_seconds=0.1,
    sr=sr
)

display(Audio(audio, rate=sr))
display(Audio(shifted_audio, rate=sr))
```

Notice that the output retains the same duration and number of samples.

The exposed region is zero-filled rather than wrapped around.

---

## Step 6 — Create a time-stretching function

We'll use librosa.

```python
def time_stretch(
    audio,
    rate=1.05
):

    return librosa.effects.time_stretch(
        audio,
        rate=rate
    )
```

### Usage

```python
slow_audio = time_stretch(
    audio,
    rate=0.95
)

fast_audio = time_stretch(
    audio,
    rate=1.05
)
```

### Listen

```python
print("Original")
display(Audio(audio, rate=sr))

print("Slower")
display(Audio(slow_audio, rate=sr))

print("Faster")
display(Audio(fast_audio, rate=sr))
```

**Understanding the rate parameter:**

- `rate < 1`: Slower and longer.
- `rate = 1`: Unchanged.
- `rate > 1`: Faster and shorter.

### Check durations

```python
print(
    "Original duration:",
    len(audio) / sr
)

print(
    "Slow duration:",
    len(slow_audio) / sr
)

print(
    "Fast duration:",
    len(fast_audio) / sr
)
```

---

## Step 7 — Create a pitch-shifting function

```python
def pitch_shift(
    audio,
    sr,
    n_steps=0.5
):

    return librosa.effects.pitch_shift(
        y=audio,
        sr=sr,
        n_steps=n_steps
    )
```

### Usage

```python
higher_pitch = pitch_shift(
    audio,
    sr,
    n_steps=0.5
)

lower_pitch = pitch_shift(
    audio,
    sr,
    n_steps=-0.5
)
```

### Listen

```python
print("Original")
display(Audio(audio, rate=sr))

print("Higher pitch")
display(Audio(higher_pitch, rate=sr))

print("Lower pitch")
display(Audio(lower_pitch, rate=sr))
```

Check whether the emotional delivery remains recognizable.

---

## Step 8 — Create a random-cropping function

```python
def random_crop(
    audio,
    sr,
    crop_seconds=3,
    rng=None
):

    if rng is None:
        rng = np.random.default_rng()

    crop_samples = int(
        crop_seconds * sr
    )

    if len(audio) < crop_samples:

        raise ValueError(
            "Audio is shorter than the requested crop."
        )

    max_start = len(audio) - crop_samples

    start = rng.integers(
        0,
        max_start + 1
    )

    end = start + crop_samples

    return audio[start:end]
```

### Usage

```python
cropped_audio = random_crop(
    audio,
    sr,
    crop_seconds=3,
    rng=rng
)

print(
    "Original duration:",
    len(audio) / sr
)

print(
    "Cropped duration:",
    len(cropped_audio) / sr
)

display(Audio(cropped_audio, rate=sr))
```

Listen and check whether the selected segment still expresses the original emotion.

**Important:** If your recording is shorter than the requested crop duration, this function raises an error instead of silently padding it. Padding should be handled separately according to your dataset strategy.

---

## Step 9 — Compare original and augmented recordings

Let's create a reusable visualization function.

We'll compare:

- Waveform.
- RMS energy.
- Duration.

```python
def compare_audio(
    original,
    augmented,
    sr,
    title="Audio Comparison"
):

    fig, axes = plt.subplots(
        2,
        2,
        figsize=(14, 8)
    )

    # Original waveform
    librosa.display.waveshow(
        original,
        sr=sr,
        ax=axes[0, 0]
    )

    axes[0, 0].set_title(
        "Original Waveform"
    )

    # Augmented waveform
    librosa.display.waveshow(
        augmented,
        sr=sr,
        ax=axes[0, 1]
    )

    axes[0, 1].set_title(
        "Augmented Waveform"
    )

    # Original RMS
    original_rms = librosa.feature.rms(
        y=original,
        frame_length=400,
        hop_length=160
    )[0]

    original_times = librosa.times_like(
        original_rms,
        sr=sr,
        hop_length=160
    )

    axes[1, 0].plot(
        original_times,
        original_rms
    )

    axes[1, 0].set_title(
        "Original RMS Energy"
    )

    axes[1, 0].set_xlabel(
        "Time (seconds)"
    )

    # Augmented RMS
    augmented_rms = librosa.feature.rms(
        y=augmented,
        frame_length=400,
        hop_length=160
    )[0]

    augmented_times = librosa.times_like(
        augmented_rms,
        sr=sr,
        hop_length=160
    )

    axes[1, 1].plot(
        augmented_times,
        augmented_rms
    )

    axes[1, 1].set_title(
        "Augmented RMS Energy"
    )

    axes[1, 1].set_xlabel(
        "Time (seconds)"
    )

    for ax in axes.flat:
        ax.set_ylabel("Amplitude")
        ax.grid()

    fig.suptitle(title)

    plt.tight_layout()
    plt.show()

    print(
        "Original duration:",
        len(original) / sr
    )

    print(
        "Augmented duration:",
        len(augmented) / sr
    )
```

### Test with noise

```python
compare_audio(
    audio,
    noisy_audio,
    sr,
    title="Original vs Noise Augmentation"
)
```

### Test with time stretching

```python
compare_audio(
    audio,
    slow_audio,
    sr,
    title="Original vs Time Stretching"
)
```

---

## Step 10 — Compare pitch before and after augmentation

This is particularly important for SER because pitch is an emotion-related feature.

We'll use the same pYIN configuration from Module 35.7.

```python
def extract_pitch_contour(
    audio,
    sr
):

    f0, voiced_flag, voiced_prob = librosa.pyin(
        audio,
        sr=sr,
        fmin=50,
        fmax=500,
        frame_length=1024,
        hop_length=160,
        fill_na=np.nan
    )

    times = librosa.times_like(
        f0,
        sr=sr,
        hop_length=160
    )

    return times, f0
```

### Compare original and pitch-shifted recordings

```python
original_times, original_f0 = extract_pitch_contour(
    audio,
    sr
)

shifted_times, shifted_f0 = extract_pitch_contour(
    higher_pitch,
    sr
)

plt.figure(figsize=(14, 5))

plt.plot(
    original_times,
    original_f0,
    label="Original"
)

plt.plot(
    shifted_times,
    shifted_f0,
    label="Pitch Shifted",
    alpha=0.7
)

plt.xlabel("Time (seconds)")
plt.ylabel("F0 (Hz)")

plt.title("Pitch Contour Comparison")

plt.legend()
plt.grid()

plt.tight_layout()
plt.show()
```

### Observe

- Did the pitch increase?
- Are the pitch contours still similar in shape?
- Did the transformation introduce unusual pitch estimates?

This helps us verify that the augmentation has the intended acoustic effect.

---

## Step 11 — Build a safe augmentation pipeline

Now let's combine our functions into a reusable augmentation pipeline.

We'll randomly select one transformation per generated sample rather than stacking every transformation together.

```python
def augment_audio(
    audio,
    sr,
    rng=None
):

    if rng is None:
        rng = np.random.default_rng()

    augmentation_choices = [
        "noise",
        "volume",
        "time_shift",
        "time_stretch",
        "pitch_shift"
    ]

    augmentation = rng.choice(
        augmentation_choices
    )

    parameters = {}

    if augmentation == "noise":

        snr_db = rng.uniform(20, 30)

        augmented = add_background_noise(
            audio,
            snr_db=snr_db,
            rng=rng
        )

        parameters["snr_db"] = snr_db

    elif augmentation == "volume":

        augmented, gain = change_volume(
            audio,
            min_gain=0.85,
            max_gain=1.15,
            rng=rng
        )

        parameters["gain"] = gain

    elif augmentation == "time_shift":

        shift = rng.uniform(-0.1, 0.1)

        augmented = time_shift(
            audio,
            shift_seconds=shift,
            sr=sr
        )

        parameters["shift_seconds"] = shift

    elif augmentation == "time_stretch":

        rate = rng.uniform(0.95, 1.05)

        augmented = time_stretch(
            audio,
            rate=rate
        )

        parameters["stretch_rate"] = rate

    elif augmentation == "pitch_shift":

        steps = rng.uniform(-0.5, 0.5)

        augmented = pitch_shift(
            audio,
            sr,
            n_steps=steps
        )

        parameters["pitch_steps"] = steps

    return augmented, augmentation, parameters
```

### Generate an augmented recording

```python
augmented_audio, augmentation_type, parameters = augment_audio(
    audio,
    sr,
    rng
)

print(
    "Applied augmentation:",
    augmentation_type
)

print(
    "Parameters:",
    parameters
)

display(Audio(audio, rate=sr))

display(
    Audio(
        augmented_audio,
        rate=sr
    )
)
```

**Important:** This pipeline is a starting experiment, not an automatic guarantee of emotion preservation. Listen to the outputs and reject unsuitable examples.

---

## Step 12 — Generate augmented training recordings

Let's assume your dataset has already been split into separate folders:

```text
audio/
    train/
    validation/
    test/
```

Only process the training folder.

For this experiment, we'll use a metadata CSV containing:

| file_path | emotion | speaker_id |
|---|---|---|
| audio/train/001.wav | happy | speaker_01 |
| audio/train/002.wav | sad | speaker_02 |
| audio/train/003.wav | angry | speaker_01 |

The labels and speaker IDs must come from your actual dataset.

### Load training metadata

```python
train_metadata = pd.read_csv(
    "train_metadata.csv"
)

train_metadata.head()
```

### Create the output folder

```python
train_folder = Path("audio/train")

augmented_folder = Path(
    "audio/train_augmented"
)

augmented_folder.mkdir(
    parents=True,
    exist_ok=True
)
```

### Generate augmented samples

```python
augmentation_metadata = []

for _, row in train_metadata.iterrows():

    file_path = Path(
        row["file_path"]
    )

    audio, sr = librosa.load(
        file_path,
        sr=SR,
        mono=True
    )

    augmented_audio, augmentation_type, parameters = augment_audio(
        audio,
        sr,
        rng
    )

    output_path = augmented_folder / (
        f"{file_path.stem}_{augmentation_type}.wav"
    )

    # Save as floating-point WAV to avoid
    # automatic integer PCM clipping.
    sf.write(
        output_path,
        augmented_audio,
        sr,
        subtype="FLOAT"
    )

    # Preserve original metadata
    augmentation_metadata.append({
        "original_filename": file_path.name,
        "augmented_filename": output_path.name,
        "file_path": str(output_path),
        "emotion": row["emotion"],
        "speaker_id": row["speaker_id"],
        "augmentation_type": augmentation_type,
        **parameters
    })
```

### Save the augmentation manifest

```python
augmentation_df = pd.DataFrame(
    augmentation_metadata
)

augmentation_df.to_csv(
    "augmentation_metadata.csv",
    index=False
)

augmentation_df.head()
```

### Important dataset engineering consideration

For a real SER project, preserve the original emotion label and speaker ID in the augmentation metadata.

Also, avoid overwriting original files.

Your dataset should retain:

- Original filename.
- Augmented filename.
- Emotion label.
- Speaker ID.
- Augmentation type.
- Transformation parameters.

For example, record the actual SNR, gain, pitch shift or stretch rate. This makes the experiment reproducible.

**One more important rule:** Do not automatically keep every augmented sample. Listen to a representative set, inspect acoustic features, and reject transformations that distort the emotional expression.

---

## 13. Final augmentation strategy for our SER project

Here is the initial strategy I recommend testing.

| Augmentation | Initial experimental range | Priority |
|---|---|---|
| Background noise | 20–30 dB SNR | Core |
| Volume variation | 0.85–1.15 gain | Core |
| Time shifting | ±100 ms | Useful |
| Time stretching | 0.95–1.05 rate | Useful |
| Pitch shifting | ±0.5 semitones | Experimental |
| Random cropping | Emotion-preserving segments | Experimental |

These are conservative starting ranges, not universally optimal values.

### What should we avoid?

- Extreme pitch shifting.
- Excessive time stretching.
- Clipping the waveform.
- Adding noise that hides speech.
- Cropping away the emotion-bearing section.
- Applying augmentation to validation and test sets.
- Creating augmented samples before splitting the original dataset.

---

## What you should remember

1. Audio augmentation increases acoustic variation in the training data.
2. Noise addition should be controlled using SNR.
3. Volume augmentation changes amplitude, not intentionally pitch or duration.
4. Time stretching changes duration while attempting to preserve pitch.
5. Pitch shifting changes pitch while attempting to preserve duration.
6. Time shifting and random cropping can remove important emotional context.
7. Every augmentation must preserve the original emotion label sufficiently.
8. Split your dataset before augmentation.
9. Keep validation and testing recordings unchanged.
10. Listen to augmented recordings instead of trusting the transformation parameters alone.
